In [197]:
# Standard
import hashlib
import json
import re
from pathlib import Path

# Data
import pandas as pd

# Imports pour les splits et fichiers.
from sklearn.model_selection import train_test_split

##
RANDOM_SEED = 42  # Reproductibilité des splits

In [ ]:
# Localise la racine du projet et définit les chemins du SFT final.

def find_project_root(start: Path | None = None) -> Path:
    """Localise la racine du projet."""
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw" / "datasets").is_dir():
            return candidate

    raise FileNotFoundError("Répertoire data/raw/datasets introuvable.")


PROJECT_ROOT = find_project_root()

OUTPUT_DIR = (
    PROJECT_ROOT / "artifacts" / "sft_dataset_creation"
)

TRIAGE_DIR = OUTPUT_DIR / "priority_pools"
FINAL_PATH = OUTPUT_DIR / "sft_final_5000.jsonl"
RUN_SPLIT_DIR = OUTPUT_DIR / "splits"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projet       : {PROJECT_ROOT}")
print(f"Pools triage : {TRIAGE_DIR}")
print(f"Final        : {FINAL_PATH}")
print(f"Splits        : {RUN_SPLIT_DIR}")



Projet       : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent
Pools triage : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\sft_dataset_creation\priority_pools
Final        : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\sft_dataset_creation\sft_final_5000.jsonl
Splits        : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\sft_dataset_creation\splits


In [143]:
# Recharge les 6 pools finaux après projection, structuration et réévaluation.

TRIAGE_FILES = {
    "fr_p1": "fr_p1_triage_revised_final.jsonl",
    "fr_p2": "fr_p2_triage_revised_final.jsonl",
    "fr_p3": "fr_p3_triage_revised_final.jsonl",
    "en_p1": "en_p1_triage_revised_final.jsonl",
    "en_p2": "en_p2_triage_revised_final.jsonl",
    "en_p3": "en_p3_triage_revised_final.jsonl",
    "en_p1_2": "en_p1_recovered_triage_revised_final.jsonl",
    "en_p2_2": "en_p3_recovered_triage_revised_final.jsonl",
    "en_p3_2": "en_p2_recovered_triage_revised_final.jsonl"

}


def load_final_pool(filename):
    """Charge et contrôle un pool final."""
    path = TRIAGE_DIR / filename
    records = read_jsonl(path)
    df = pd.DataFrame(records)

    assert df["id"].is_unique
    assert df["final_priority"].isin({"P1", "P2", "P3"}).all()

    print(f"✓ {filename:<45} {len(df):>6,}")
    return df


fr_p1 = load_final_pool(TRIAGE_FILES["fr_p1"])
fr_p2 = load_final_pool(TRIAGE_FILES["fr_p2"])
fr_p3 = load_final_pool(TRIAGE_FILES["fr_p3"])

en_p1 = load_final_pool(TRIAGE_FILES["en_p1"])
en_p2 = load_final_pool(TRIAGE_FILES["en_p2"])
en_p3 = load_final_pool(TRIAGE_FILES["en_p3"])

en_p1_2 = load_final_pool(TRIAGE_FILES["en_p1_2"])
en_p2_2 = load_final_pool(TRIAGE_FILES["en_p2_2"])
en_p3_2 = load_final_pool(TRIAGE_FILES["en_p3_2"])

✓ fr_p1_triage_revised_final.jsonl               1,670
✓ fr_p2_triage_revised_final.jsonl               1,473
✓ fr_p3_triage_revised_final.jsonl               9,297
✓ en_p1_triage_revised_final.jsonl                  79
✓ en_p2_triage_revised_final.jsonl                 105
✓ en_p3_triage_revised_final.jsonl               1,316
✓ en_p1_recovered_triage_revised_final.jsonl        26
✓ en_p3_recovered_triage_revised_final.jsonl     4,479
✓ en_p2_recovered_triage_revised_final.jsonl        78


In [144]:
# Consolide les 18 523 records réévalués.

triage_master = pd.concat(
    [
        fr_p1, fr_p2, fr_p3,
        en_p1, en_p2, en_p3,
        en_p1_2, en_p2_2, en_p3_2
    ],
    ignore_index=True
)

assert len(triage_master) == 18_523
assert triage_master["id"].is_unique
assert triage_master["language"].isin({"fr", "en"}).all()
assert triage_master["final_priority"].isin({"P1", "P2", "P3"}).all()
assert triage_master["triage_rule_version"].eq(
    "hospital_severity_risk_v1"
).all()

print(f"✓ Master consolidé : {len(triage_master):,}")
print(f"✓ IDs uniques      : {triage_master['id'].nunique():,}")
print("✓ Règle triage     : hospital_severity_risk_v1")

display(
    triage_master
    .groupby(["language", "final_priority"])
    .size()
    .unstack(fill_value=0)
)

✓ Master consolidé : 18,523
✓ IDs uniques      : 18,523
✓ Règle triage     : hospital_severity_risk_v1


final_priority,P1,P2,P3
language,,,
en,922,2504,2657
fr,1491,3362,7587


In [145]:
# JSONL

def read_jsonl(path: Path) -> list[dict]:
    """Charge un JSONL."""
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def write_jsonl(path: Path, records: list[dict]) -> None:
    """Écrit un JSONL."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


In [146]:
# Sauvegarde le master consolidé avant audit.

TRIAGE_MASTER_PATH = (
    OUTPUT_DIR / "triage_master_18523.jsonl"
)

write_jsonl(
    TRIAGE_MASTER_PATH,
    triage_master.to_dict("records")
)

# Vérification après sauvegarde.

saved_master = read_jsonl(TRIAGE_MASTER_PATH)

assert len(saved_master) == 18_523
assert len({r["id"] for r in saved_master}) == 18_523

print(f"✓ Master sauvegardé : {len(saved_master):,} records")
print(f"✓ IDs uniques       : {len({r['id'] for r in saved_master}):,}")
print(f"✓ Fichier           : {TRIAGE_MASTER_PATH}")

✓ Master sauvegardé : 18,523 records
✓ IDs uniques       : 18,523
✓ Fichier           : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\sft_dataset_creation\triage_master_18523.jsonl


In [147]:
# Champs réellement visibles par le modèle.

PATIENT_INPUT_FIELDS = (
    "age_days", "age_months", "age_years", "sex", "symptoms",
    "condition_mentions", "medical_history", "duration",
    "evolution", "critical_signs", "vital_signs"
)

KNOWLEDGE_INPUT_FIELDS = (
    "symptoms", "conditions", "complications", "diagnostic_tests",
    "laboratory_findings", "imaging_findings", "risk_factors",
    "treatments", "medications", "procedures", "pathogens", "anatomy"
)

# Normalise les valeurs et retire la mécanique QCM.

def clinical_text(value) -> str:
    """Convertit une valeur clinique en texte."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False) if value else ""
    return str(value).strip()


QCM_OPTION_RE = re.compile(r"(?m)^\s*(?:[A-E][.)]|[1-9][.)])\s+")
QCM_QUESTION_RE = re.compile(
    r"(?i)(?:quel(?:le|les|s)?\b|indiquez\b|choisissez\b|cochez\b|"
    r"sélectionnez\b|which\b|choose\b|select\b)"
)


def clean_clinical_narrative(value) -> str:
    """Nettoie le récit clinique des éléments QCM."""
    text = re.sub(
        r"^\s*(?:input|context)\s*:\s*", "",
        clinical_text(value), flags=re.I
    )

    option = QCM_OPTION_RE.search(text)
    if not option:
        return text.strip()

    lines = text[:option.start()].rstrip().splitlines()

    while lines and (
        not lines[-1].strip()
        or lines[-1].strip().endswith("?")
        or QCM_QUESTION_RE.match(lines[-1].strip())
    ):
        lines.pop()

    return "\n".join(lines).strip()


# Construit exactement l'information clinique visible par le modèle.

def structured_information(record, key, fields) -> list[str]:
    """Extrait les champs structurés autorisés."""
    data = record.get(key) or {}
    if not isinstance(data, dict):
        return []

    return [
        f"{field}: {clinical_text(data[field])}"
        for field in fields
        if clinical_text(data.get(field))
    ]

def language_of(record: dict) -> str:
    """Retourne fr ou en."""
    language = str(
        record.get("language")
        or record.get("detected_language")
        or "fr"
    ).lower().strip()

    return "en" if language.startswith("en") else "fr"

PLACEHOLDER_NARRATIVES = {
    "le patient présente la situation médicale décrite.",
    "the patient presents with the described medical condition."
}


def valid_narrative(value) -> str:
    """Conserve uniquement un récit clinique informatif."""
    text = clean_clinical_narrative(value)

    if text.lower() in PLACEHOLDER_NARRATIVES:
        return ""

    return text


def build_model_context(record: dict) -> str:
    """Construit l'entrée clinique visible par le modèle."""
    sections = []

    narrative = (
        valid_narrative(record.get("synthetic_patient"))
        or valid_narrative(record.get("input"))
        or valid_narrative(record.get("context"))
    )

    if narrative:
        sections.append(f"Contexte médical:\n{narrative}")

    patient = structured_information(
        record, "patient", PATIENT_INPUT_FIELDS
    )

    knowledge = structured_information(
        record, "medical_knowledge", KNOWLEDGE_INPUT_FIELDS
    )

    if patient:
        sections.append("Informations patient:\n" + "\n".join(patient))

    if knowledge:
        sections.append("Informations médicales:\n" + "\n".join(knowledge))

    return "\n\n".join(sections).strip()

def priority_of(record: dict) -> int:
    """Normalise la priorité finale en entier."""
    priority = str(record["final_priority"]).upper().replace("P", "").strip()
    priority = int(priority)

    if priority not in {1, 2, 3}:
        raise ValueError(f"Priorité invalide : {record.get('id')}")

    return priority


def normalize_sft(record: dict) -> dict:
    """Normalise un candidat SFT."""
    return {
        **record,
        "language": language_of(record),
        "priority": priority_of(record),
        "model_context": build_model_context(record)
    }
        
# Hash uniquement l'entrée visible par le modèle.

def normalize_for_hash(text: str) -> str:
    """Normalise le texte avant comparaison."""
    return re.sub(r"\s+", " ", clinical_text(text)).strip().lower()


def text_hash(text: str) -> str:
    """Calcule le hash SHA-256 du contexte."""
    return hashlib.sha256(
        normalize_for_hash(text).encode("utf-8")
    ).hexdigest()

In [148]:
# Normalise les candidats puis isole les contextes vides.

audit_master = pd.DataFrame(
    normalize_sft(record)
    for record in triage_master.to_dict("records")
)

assert len(audit_master) == 18_523
assert audit_master["id"].is_unique
assert audit_master["language"].isin({"fr", "en"}).all()
assert audit_master["priority"].isin({1, 2, 3}).all()


# Isole les entrées sans information clinique exploitable.

empty_mask = (
    audit_master["model_context"]
    .fillna("")
    .str.strip()
    .eq("")
)

empty_context_df = audit_master.loc[empty_mask].copy()

audit_master = (
    audit_master.loc[~empty_mask]
    .copy()
    .reset_index(drop=True)
)

assert audit_master["model_context"].str.strip().ne("").all()


# Hash uniquement les contextes exploitables.

audit_master["context_hash"] = (
    audit_master["model_context"].map(text_hash)
)

print(f"✓ Records initiaux     : {len(triage_master):,}")
print(f"✓ Contextes vides      : {len(empty_context_df):,}")
print(f"✓ Records exploitables : {len(audit_master):,}")
print(f"✓ Contextes uniques    : {audit_master['context_hash'].nunique():,}")
print(
    f"✓ Doublons potentiels  : "
    f"{len(audit_master) - audit_master['context_hash'].nunique():,}"
)

✓ Records initiaux     : 18,523
✓ Contextes vides      : 875
✓ Records exploitables : 17,648
✓ Contextes uniques    : 14,915
✓ Doublons potentiels  : 2,733


In [149]:
# Isole les records sans entrée clinique exploitable.

empty_mask = (
    audit_master["model_context"]
    .fillna("")
    .str.strip()
    .eq("")
)

empty_context_df = audit_master.loc[empty_mask].copy()

audit_master = (
    audit_master.loc[~empty_mask]
    .copy()
    .reset_index(drop=True)
)

print(f"✓ Contextes vides : {len(empty_context_df):,}")
print(f"✓ Candidats audit : {len(audit_master):,}")

✓ Contextes vides : 0
✓ Candidats audit : 17,648


In [150]:
# Exclut les conflits de priorité puis déduplique les contextes identiques.

priority_counts = (
    audit_master.groupby("context_hash")["priority"].nunique()
)

conflict_hashes = set(
    priority_counts[priority_counts.gt(1)].index
)

conflicts_df = audit_master[
    audit_master["context_hash"].isin(conflict_hashes)
].copy()

clean_df = audit_master[
    ~audit_master["context_hash"].isin(conflict_hashes)
].copy()

before = len(clean_df)

clean_df = (
    clean_df
    .sort_values("id")
    .drop_duplicates("context_hash", keep="first")
    .reset_index(drop=True)
)

assert clean_df["context_hash"].is_unique
assert not clean_df["context_hash"].isin(conflict_hashes).any()

print(
    f"✓ Conflits exclus  : {len(conflict_hashes):,} groupes / "
    f"{len(conflicts_df):,} lignes"
)
print(f"✓ Doublons retirés : {before - len(clean_df):,}")
print(f"✓ Candidats propres: {len(clean_df):,}")

✓ Conflits exclus  : 166 groupes / 758 lignes
✓ Doublons retirés : 2,141
✓ Candidats propres: 14,749


In [151]:
# Audite les contenus HPO sans les exclure.

hpo_mask = clean_df["model_context"].str.contains(
    r"\bHuman Phenotype Ontology\b|\bHPO\b",
    case=False,
    regex=True,
    na=False
)

hpo_df = clean_df.loc[
    hpo_mask,
    ["id", "language", "priority", "model_context"]
].copy()

print(f"✓ HPO détectés : {len(hpo_df):,}")

✓ HPO détectés : 915


In [152]:
# Marque les contenus HPO pour l'audit qualité.

clean_df["is_hpo"] = clean_df["model_context"].str.contains(
    r"\bHuman Phenotype Ontology\b|\bHPO\b",
    case=False,
    regex=True,
    na=False
)

print(f"✓ HPO détectés : {clean_df['is_hpo'].sum():,}")
print(f"✓ Pool conservé : {len(clean_df):,}")

✓ HPO détectés : 915
✓ Pool conservé : 14,749


In [153]:
# # Charge l'extracteur clinique utilisé pour l'audit qualité.

# import sys

# if str(PROJECT_ROOT) not in sys.path:
#     sys.path.insert(0, str(PROJECT_ROOT))

# from scr.triage.extraction_general import GeneralMedicalExtractor
# from scr.triage.knowledge_matching import build_french_motif_lexicon
# from scr.triage.protocol import load_triage_protocol

# PROTOCOL_PATH = PROJECT_ROOT / "data" / "raw" / "triage_protocol.json"

# TRIAGE_PROTOCOL = load_triage_protocol(PROTOCOL_PATH)
# FRENCH_MOTIF_LEXICON = build_french_motif_lexicon(TRIAGE_PROTOCOL)
# clinical_extractor = GeneralMedicalExtractor(FRENCH_MOTIF_LEXICON)

# print(f"✓ Motifs chargés : {len(FRENCH_MOTIF_LEXICON):,}")

In [154]:
# import importlib
# import scr.triage.medical_knowledge
# import scr.triage.extraction_general

# # Recharge les scripts modifiés.
# importlib.reload(scr.triage.medical_knowledge_patterns)
# importlib.reload(scr.triage.medical_knowledge)
# importlib.reload(scr.triage.extraction_general)

# # Réimporte la classe rechargée.
# from scr.triage.extraction_general import GeneralMedicalExtractor

# # Recrée l'extracteur avec le nouveau code.
# clinical_extractor = GeneralMedicalExtractor(FRENCH_MOTIF_LEXICON)

In [155]:
# # Test EN : conserve question/answer et lance l'extracteur complet.

# test_en = sft_df.loc[sft_df["language"].eq("en")].head(3)

# for _, row in test_en.iterrows():
#     record = row.to_dict()
#     result = clinical_extractor.extract(record)
#     knowledge = result["medical_knowledge"]

#     print("=" * 100)
#     print("ID       :", record["id"])
#     print("QUESTION :", record["question"])
#     print("ANSWER   :", str(record["answer"])[:500])
#     print("INTENT   :", knowledge.get("intent"))

#     print("\nMEDICAL KNOWLEDGE")

#     for field in (
#         "conditions",
#         "symptoms",
#         "treatments",
#         "medications",
#         "procedures",
#         "laboratory_findings",
#         "imaging_findings",
#         "diagnostic_tests",
#         "complications",
#         "risk_factors",
#         "pathogens",
#         "causes",
#         "anatomy"
#     ):
#         values = knowledge.get(field)

#         if values:
#             print(f"{field} : {values}")

In [156]:
# import pandas as pd
# from tqdm.auto import tqdm


# # Extrait tous les EN avec suivi de progression.
# # Extrait tous les EN avec suivi de progression.
# en_results = []
# en_extracted = []

# en_records = sft_df.loc[sft_df["language"].eq("en")].to_dict("records")

# for row in tqdm(en_records, desc="Extraction EN", unit="record"):
#     result = clinical_extractor.extract(row)
#     knowledge = result["medical_knowledge"]

#     # Record complet enrichi.
#     en_extracted.append(result)

#     # Audit compact.
#     en_results.append({
#         "id": row["id"],
#         "intent": knowledge.get("intent"),
#         "question_subject": knowledge.get("question_subject"),
#         "fields": knowledge.get("extracted_fields", []),
#         "has_knowledge": bool(knowledge.get("extracted_fields"))
#     })

# en_audit = pd.DataFrame(en_results)


# # Mesure la couverture par intention.
# coverage = (
#     en_audit.groupby("intent", dropna=False)
#     .agg(
#         records=("id", "size"),
#         extracted=("has_knowledge", "sum")
#     )
# )

# coverage["coverage_pct"] = (
#     coverage["extracted"]
#     .div(coverage["records"])
#     .mul(100)
#     .round(1)
# )

# display(coverage.sort_values("records", ascending=False))

In [157]:
# EN_EXTRACTED_PATH = OUTPUT_DIR/ "clinical_extracted_en.jsonl"

# with EN_EXTRACTED_PATH.open("w", encoding="utf-8") as f:
#     for record in en_extracted:
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")

# print(f"Records EN : {len(en_extracted):,}")
# print(f"Fichier    : {EN_EXTRACTED_PATH}")

In [158]:
# EN_AUDIT_PATH = OUTPUT_DIR / "clinical_extraction_en_audit.csv"

# en_audit.to_csv(
#     EN_AUDIT_PATH,
#     index=False
# )

# print(f"Audit : {EN_AUDIT_PATH}")

In [159]:
# # Réintègre les extractions EN dans le pool général.
# en_extracted_df = pd.DataFrame(en_extracted).set_index("id")
# sft_df = sft_df.set_index("id")

# for col in ("patient", "medical_knowledge", "medical_extraction"):
#     sft_df.loc[en_extracted_df.index, col] = en_extracted_df[col]

# sft_df = sft_df.reset_index()

In [160]:
# Contrôle le pool propre et crée les 6 catégories langue × priorité.

assert clean_df["id"].is_unique
assert clean_df["language"].isin({"fr", "en"}).all()
assert clean_df["priority"].isin({1, 2, 3}).all()
assert clean_df["model_context"].str.strip().ne("").all()

# Construit la catégorie langue × priorité.

clean_df["category"] = (
    clean_df["language"].str.upper()
    + "_P"
    + clean_df["priority"].astype(str)
)

categories = [
    "FR_P1", "FR_P2", "FR_P3",
    "EN_P1", "EN_P2", "EN_P3"
]

assert set(clean_df["category"]) == set(categories)

distribution = (
    clean_df["category"]
    .value_counts()
    .reindex(categories, fill_value=0)
    .rename("records")
    .to_frame()
)

display(distribution)

print(f"✓ Pool propre : {len(clean_df):,} candidats")
print("✓ 6 catégories langue × priorité présentes")

,records
category,
FR_P1,1105
FR_P2,2363
FR_P3,5241
EN_P1,921
EN_P2,2480
EN_P3,2639


✓ Pool propre : 14,749 candidats
✓ 6 catégories langue × priorité présentes


In [161]:
# Mesure la richesse patient et des connaissances médicales.
# Mesure la richesse patient et des connaissances médicales.

CATEGORIES = (
    "FR_P1", "FR_P2", "FR_P3",
    "EN_P1", "EN_P2", "EN_P3"
)

PATIENT_QUALITY_FIELDS = (
    "sex", "symptoms", "condition_mentions", "medical_history",
    "duration", "evolution", "critical_signs", "vital_signs"
)

KNOWLEDGE_QUALITY_FIELDS = (
    "symptoms", "conditions", "causes", "complications",
    "diagnostic_tests", "laboratory_findings", "imaging_findings",
    "risk_factors", "treatments", "medications", "procedures",
    "pathogens", "anatomy"
)

def has_information(value) -> bool:
    """Teste si une valeur est renseignée."""
    if value is None:
        return False
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (list, tuple, dict)):
        return bool(value)
    return True


def quality_score(data, fields, age=False) -> tuple[int, str]:
    """Mesure la richesse d'un bloc clinique."""
    if not isinstance(data, dict):
        return 0, "none"

    score = sum(
        has_information(data.get(field))
        for field in fields
    )

    if age:
        score += any(
            has_information(data.get(field))
            for field in ("age_days", "age_months", "age_years")
        )

    quality = (
        "strong" if score >= 4
        else "medium" if score >= 2
        else "weak" if score == 1
        else "none"
    )

    return score, quality


# Calcule les deux dimensions de qualité.

patient = clean_df["patient"].apply(
    lambda x: quality_score(x, PATIENT_QUALITY_FIELDS, age=True)
)

knowledge = clean_df["medical_knowledge"].apply(
    lambda x: quality_score(x, KNOWLEDGE_QUALITY_FIELDS)
)

clean_df["patient_score"] = patient.str[0]
clean_df["patient_quality"] = patient.str[1]

clean_df["knowledge_score"] = knowledge.str[0]
clean_df["knowledge_quality"] = knowledge.str[1]

clean_df["has_patient"] = clean_df["patient_score"].gt(0)
clean_df["has_knowledge"] = clean_df["knowledge_score"].gt(0)


# Résume les deux dimensions par catégorie.

quality_summary = (
    clean_df.groupby("category")
    .agg(
        records=("id", "size"),
        patient_mean=("patient_score", "mean"),
        patient_strong=("patient_quality", lambda x: x.eq("strong").sum()),
        patient_medium=("patient_quality", lambda x: x.eq("medium").sum()),
        patient_weak=("patient_quality", lambda x: x.eq("weak").sum()),
        patient_none=("patient_quality", lambda x: x.eq("none").sum()),
        knowledge_mean=("knowledge_score", "mean"),
        knowledge_strong=("knowledge_quality", lambda x: x.eq("strong").sum()),
        knowledge_medium=("knowledge_quality", lambda x: x.eq("medium").sum()),
        knowledge_weak=("knowledge_quality", lambda x: x.eq("weak").sum()),
        knowledge_none=("knowledge_quality", lambda x: x.eq("none").sum())
    )
    .reindex(CATEGORIES)
    .round(2)
)

display(quality_summary)

print(f"✓ Candidats analysés       : {len(clean_df):,}")
print(f"✓ Patient renseigné        : {clean_df['has_patient'].sum():,}")
print(f"✓ Connaissances renseignées: {clean_df['has_knowledge'].sum():,}")

,records,patient_mean,patient_strong,patient_medium,patient_weak,patient_none,knowledge_mean,knowledge_strong,knowledge_medium,knowledge_weak,knowledge_none
category,,,,,,,,,,,
FR_P1,1105,2.66,361,410,172,162,1.45,79,416,270,340
FR_P2,2363,1.98,550,746,340,727,1.32,114,825,737,687
FR_P3,5241,1.72,797,1836,1048,1560,0.99,106,1300,1784,2051
EN_P1,921,1.76,0,630,291,0,2.57,247,409,201,64
EN_P2,2480,1.46,0,1121,1359,0,2.21,552,904,659,365
EN_P3,2639,1.53,0,1383,1256,0,1.97,442,1015,680,502


✓ Candidats analysés       : 14,749
✓ Patient renseigné        : 12,300
✓ Connaissances renseignées: 10,740


In [162]:
# Contrôle le pool anglais déjà projeté, structuré et réévalué.

en_df = (
    clean_df.loc[clean_df["language"].eq("en")]
    .copy()
    .reset_index(drop=True)
)

assert en_df["id"].is_unique
assert en_df["priority"].isin({1, 2, 3}).all()
assert en_df["category"].isin({"EN_P1", "EN_P2", "EN_P3"}).all()

en_distribution = (
    en_df["category"]
    .value_counts()
    .reindex(["EN_P1", "EN_P2", "EN_P3"], fill_value=0)
    .rename("records")
    .to_frame()
)

display(en_distribution)

print(f"✓ Pool EN : {len(en_df):,} candidats")

,records
category,
EN_P1,921
EN_P2,2480
EN_P3,2639


✓ Pool EN : 6,040 candidats


In [163]:
QUOTAS = {
    "FR_P1": 834,
    "FR_P2": 833,
    "FR_P3": 833,
    "EN_P1": 834,
    "EN_P2": 833,
    "EN_P3": 833
}

assert sum(QUOTAS.values()) == 5_000

In [164]:
# Prépare les critères de qualité pour la sélection finale.

QUALITY_RANK = {
    "strong": 3,
    "medium": 2,
    "weak": 1,
    "none": 0
}

CONFIDENCE_RANK = {
    "high": 3,
    "medium": 2,
    "low": 1
}


def triage_value(value, key, default=None):
    """Lit une valeur du triage historique."""
    return value.get(key, default) if isinstance(value, dict) else default


# Qualité clinique.

clean_df["patient_rank"] = (
    clean_df["patient_quality"]
    .map(QUALITY_RANK)
    .fillna(0)
    .astype(int)
)

clean_df["knowledge_rank"] = (
    clean_df["knowledge_quality"]
    .map(QUALITY_RANK)
    .fillna(0)
    .astype(int)
)


# Confiance historique du triage.

clean_df["triage_confidence"] = clean_df["triage_final"].apply(
    lambda x: str(triage_value(x, "confidence", "")).lower().strip()
)

clean_df["confidence_score"] = clean_df["triage_final"].apply(
    lambda x: float(triage_value(x, "confidence_score", 0) or 0)
)

clean_df["decision_method"] = clean_df["triage_final"].apply(
    lambda x: str(triage_value(x, "decision_method", "")).strip()
)

clean_df["confidence_rank"] = (
    clean_df["triage_confidence"]
    .map(CONFIDENCE_RANK)
    .fillna(0)
    .astype(int)
)

print(f"✓ Candidats classables : {len(clean_df):,}")

✓ Candidats classables : 14,749


In [165]:
# Contrôle les méthodes historiques sans influencer fortement la sélection.

clean_df["decision_method"] = clean_df["triage_final"].apply(
    lambda x: str(triage_value(x, "decision_method", "")).strip()
)

method_summary = (
    clean_df["decision_method"]
    .replace("", "unknown")
    .value_counts()
    .rename("records")
    .to_frame()
)

display(method_summary)

print(
    f"✓ Méthodes renseignées : "
    f"{clean_df['decision_method'].ne('').sum():,}/{len(clean_df):,}"
)

,records
decision_method,
protocol_review,5424
knowledge_reference,3280
routing_review,2684
deterministic_unique,1238
exact_rule,829
deterministic_same_priority,553
deterministic_multi_evidence_dominance,354
deterministic_evidence_dominance,276
exact_rule_convergence,111


✓ Méthodes renseignées : 14,749/14,749


In [166]:
# Classe les candidats selon la qualité de l'entrée SFT.

clean_df["context_length"] = (
    clean_df["model_context"]
    .fillna("")
    .str.len()
)

clean_df["quality_total"] = (
    clean_df["patient_score"]
    + clean_df["knowledge_score"]
)

clean_df["hpo_penalty"] = (
    clean_df["is_hpo"].astype(int)
)

RANK_COLUMNS = [
    "quality_total",
    "patient_score",
    "knowledge_score",
    "hpo_penalty",
    "confidence_rank",
    "context_length",
    "id"
]

RANK_ASCENDING = [
    False,
    False,
    False,
    True,
    False,
    False,
    True
]


print(f"✓ Candidats classables : {len(clean_df):,}")

✓ Candidats classables : 14,749


In [167]:
# Sélectionne les meilleurs candidats de chaque catégorie.

selected_parts = [
    clean_df.loc[clean_df["category"].eq(category)]
    .sort_values(RANK_COLUMNS, ascending=RANK_ASCENDING)
    .head(quota)
    for category, quota in QUOTAS.items()
]

sft_selected = (
    pd.concat(selected_parts, ignore_index=True)
    .reset_index(drop=True)
)

assert len(sft_selected) == 5_000
assert sft_selected["id"].is_unique
assert sft_selected["context_hash"].is_unique

display(
    sft_selected["category"]
    .value_counts()
    .reindex(QUOTAS)
    .rename("selected")
    .to_frame()
)

print(f"✓ SFT sélectionné : {len(sft_selected):,}")

,selected
category,
FR_P1,834
FR_P2,833
FR_P3,833
EN_P1,834
EN_P2,833
EN_P3,833


✓ SFT sélectionné : 5,000


In [168]:
# Audite la qualité des 6 groupes sélectionnés.

selection_audit = (
    sft_selected.groupby("category")
    .agg(
        records=("id", "size"),
        patient_strong=("patient_quality", lambda x: x.eq("strong").sum()),
        patient_medium=("patient_quality", lambda x: x.eq("medium").sum()),
        patient_weak=("patient_quality", lambda x: x.eq("weak").sum()),
        patient_none=("patient_quality", lambda x: x.eq("none").sum()),
        knowledge_strong=("knowledge_quality", lambda x: x.eq("strong").sum()),
        knowledge_medium=("knowledge_quality", lambda x: x.eq("medium").sum()),
        knowledge_weak=("knowledge_quality", lambda x: x.eq("weak").sum()),
        knowledge_none=("knowledge_quality", lambda x: x.eq("none").sum()),
        patient_mean=("patient_score", "mean"),
        knowledge_mean=("knowledge_score", "mean"),
        context_mean=("context_length", "mean"),
        hpo=("is_hpo", "sum")
    )
    .reindex(CATEGORIES)
    .round(2)
)

display(selection_audit)

print(f"✓ Records sélectionnés : {len(sft_selected):,}")
print(f"✓ IDs uniques          : {sft_selected['id'].nunique():,}")
print(f"✓ Contextes uniques    : {sft_selected['context_hash'].nunique():,}")

,records,patient_strong,patient_medium,patient_weak,patient_none,knowledge_strong,knowledge_medium,knowledge_weak,knowledge_none,patient_mean,knowledge_mean,context_mean,hpo
category,,,,,,,,,,,,,
FR_P1,834,361,410,59,4,79,407,216,132,3.38,1.83,846.52,0
FR_P2,833,541,278,14,0,114,553,134,32,3.87,2.32,760.55,0
FR_P3,833,683,149,1,0,98,543,184,8,4.10,2.34,808.90,0
EN_P1,834,0,630,204,0,247,409,154,24,1.84,2.79,880.96,14
EN_P2,833,0,648,185,0,552,276,5,0,1.80,4.10,1099.72,2
EN_P3,833,0,720,113,0,442,389,2,0,1.88,3.75,900.64,2


✓ Records sélectionnés : 5,000
✓ IDs uniques          : 5,000
✓ Contextes uniques    : 5,000


In [170]:
# Contrôle et sauvegarde les 5 000 records sélectionnés.

EXPECTED = {
    "FR_P1": 834,
    "FR_P2": 833,
    "FR_P3": 833,
    "EN_P1": 834,
    "EN_P2": 833,
    "EN_P3": 833
}

distribution = (
    sft_selected["category"]
    .value_counts()
    .reindex(EXPECTED, fill_value=0)
)

final_priority = (
    sft_selected["final_priority"]
    .astype(str)
    .str.upper()
    .str.replace("P", "", regex=False)
    .astype(int)
)

assert len(sft_selected) == 5_000
assert sft_selected["id"].is_unique
assert sft_selected["context_hash"].is_unique
assert distribution.to_dict() == EXPECTED
assert sft_selected["model_context"].str.strip().ne("").all()
assert sft_selected["priority"].isin({1, 2, 3}).all()
assert sft_selected["priority"].eq(final_priority).all()

write_jsonl(
    FINAL_PATH,
    sft_selected.to_dict("records")
)

display(distribution.rename("records").to_frame())

print(f"✓ Records          : {len(sft_selected):,}")
print(f"✓ IDs uniques      : {sft_selected['id'].nunique():,}")
print(f"✓ Contextes uniques: {sft_selected['context_hash'].nunique():,}")
print(f"✓ FR               : {sft_selected['language'].eq('fr').sum():,}")
print(f"✓ EN               : {sft_selected['language'].eq('en').sum():,}")
print(f"✓ Priorités        : 1 / 2 / 3")
print(f"✓ Sauvegarde       : {FINAL_PATH.resolve()}")

,records
category,
FR_P1,834
FR_P2,833
FR_P3,833
EN_P1,834
EN_P2,833
EN_P3,833


✓ Records          : 5,000
✓ IDs uniques      : 5,000
✓ Contextes uniques: 5,000
✓ FR               : 2,500
✓ EN               : 2,500
✓ Priorités        : 1 / 2 / 3
✓ Sauvegarde       : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\sft_dataset_creation\sft_final_5000.jsonl


In [ ]:
# # Recharge les 5 000 records sélectionnés.

# saved_records = read_jsonl(SFT_SELECTED_PATH)
# sft_selected = pd.DataFrame(saved_records)

# assert len(sft_selected) == 5_000
# assert sft_selected["id"].is_unique

# print(f"✓ SFT sélectionné       : {len(sft_selected):,} records")


# # Recharge les 1 500 EN avec projection patient corrigée.

# EN_PROJECTED_PATH = (
#     OUTPUT_DIR / "sft_selected_1500_en_projected.jsonl"
# )

# en_projected_records = read_jsonl(EN_PROJECTED_PATH)
# en_projected = pd.DataFrame(en_projected_records)

# assert len(en_projected) == 1_500
# assert en_projected["id"].is_unique
# assert en_projected["language"].eq("en").all()
# assert (
#     en_projected["synthetic_patient"]
#     .fillna("")
#     .str.strip()
#     .ne("")
#     .all()
# )

# print(f"✓ EN projetés corrigés  : {len(en_projected):,} records")


# # Recharge les 1 500 EN avec patient restructuré.

# EN_PATIENT_PATH = (
#     OUTPUT_DIR / "sft_selected_1500_en_patient_structured.jsonl"
# )

# en_patient_records = read_jsonl(EN_PATIENT_PATH)
# en_patient = pd.DataFrame(en_patient_records)

# assert len(en_patient) == 1_500
# assert en_patient["id"].is_unique
# assert en_patient["language"].eq("en").all()
# assert en_patient["patient"].apply(
#     lambda x: isinstance(x, dict) and bool(x)
# ).all()

# print(f"✓ EN patient structurés : {len(en_patient):,} records")


# # Vérifie que les trois fichiers concernent exactement les mêmes 1 500 EN.

# selected_en_ids = set(
#     sft_selected.loc[
#         sft_selected["language"].eq("en"),
#         "id"
#     ]
# )

# assert selected_en_ids == set(en_projected["id"])
# assert selected_en_ids == set(en_patient["id"])

# print("✓ Les 1 500 IDs EN correspondent dans les 3 fichiers")

# # Recharge les 5 000 réévalués.

# SFT_REEVALUATED_PATH = (
#     OUTPUT_DIR / "sft_5000_priority_reevaluated.jsonl"
# )

# sft_reevaluated_records = read_jsonl(
#     SFT_REEVALUATED_PATH
# )

# sft_reevaluated = pd.DataFrame(
#     sft_reevaluated_records
# )

# assert len(sft_reevaluated) == 5_000
# assert sft_reevaluated["id"].is_unique

# print(
#     f"✓ SFT réévalué          : "
#     f"{len(sft_reevaluated):,} records"
# )


# # Recharge le pool SFT rich réévalué.

# SFT_RICH_REEVALUATED_PATH = (
#     OUTPUT_DIR / "sft_rich_priority_full_input_reevaluated.jsonl"
# )

# sft_rich_reevaluated_records = read_jsonl(
#     SFT_RICH_REEVALUATED_PATH
# )

# sft_rich_reevaluated = pd.DataFrame(
#     sft_rich_reevaluated_records
# )

# assert len(sft_rich_reevaluated) == 18_523
# assert sft_rich_reevaluated["id"].is_unique

# print(
#     f"✓ SFT rich réévalué     : "
#     f"{len(sft_rich_reevaluated):,} records"
# )

In [ ]:
# # Construit le MASTER des 18 523 avec toutes les améliorations disponibles.

# master = sft_rich_reevaluated.copy()

# assert len(master) == 18_523
# assert master["id"].is_unique


# # Sources d'amélioration par ID.

# projected_map = (
#     en_projected
#     .set_index("id")["synthetic_patient"]
#     .to_dict()
# )

# patient_map = (
#     en_patient
#     .set_index("id")["patient"]
#     .to_dict()
# )


# # Identifie les colonnes de réévaluation des 5 000.

# PRIORITY_CANDIDATES = (
#     "new_priority_full_input",
#     "new_priority",
#     "verified_priority"
# )

# EXPLANATION_CANDIDATES = (
#     "new_explanation_full_input",
#     "new_explanation",
#     "verified_explanation"
# )


# def find_column(df, candidates):
#     """Retourne la première colonne disponible."""
#     return next(
#         (col for col in candidates if col in df.columns),
#         None
#     )


# priority_col = find_column(
#     sft_reevaluated,
#     PRIORITY_CANDIDATES
# )

# explanation_col = find_column(
#     sft_reevaluated,
#     EXPLANATION_CANDIDATES
# )

# assert priority_col is not None, (
#     "Aucune colonne de nouvelle priorité trouvée"
# )

# assert explanation_col is not None, (
#     "Aucune colonne de nouvelle explication trouvée"
# )

# print(f"✓ Priorité trouvée    : {priority_col}")
# print(f"✓ Explication trouvée : {explanation_col}")


# # Maps de la réévaluation récente des 5 000.

# priority_map = (
#     sft_reevaluated
#     .set_index("id")[priority_col]
#     .to_dict()
# )

# explanation_map = (
#     sft_reevaluated
#     .set_index("id")[explanation_col]
#     .to_dict()
# )

# assert len(priority_map) == 5_000
# assert len(explanation_map) == 5_000


# # Injecte la projection EN disponible.

# if "synthetic_patient" not in master.columns:
#     master["synthetic_patient"] = None

# projected = master["id"].map(projected_map)

# master["synthetic_patient"] = (
#     projected.combine_first(
#         master["synthetic_patient"]
#     )
# )


# # Remplace patient uniquement pour les 1 500 EN restructurés.

# mask_patient = master["id"].isin(patient_map)

# master.loc[
#     mask_patient,
#     "patient"
# ] = (
#     master.loc[mask_patient, "id"]
#     .map(patient_map)
# )


# # Conserve la réévaluation du pool 18 523.

# if "new_priority_full_input" not in master.columns:
#     master["new_priority_full_input"] = None

# if "new_explanation_full_input" not in master.columns:
#     master["new_explanation_full_input"] = None


# # Les 5 000 sélectionnés prennent leur réévaluation la plus récente.

# latest_priority = master["id"].map(priority_map)
# latest_explanation = master["id"].map(explanation_map)

# master["new_priority_full_input"] = (
#     latest_priority.combine_first(
#         master["new_priority_full_input"]
#     )
# )

# master["new_explanation_full_input"] = (
#     latest_explanation.combine_first(
#         master["new_explanation_full_input"]
#     )
# )


# # Trace l'état réel de chaque record.

# selected_ids = set(sft_selected["id"])

# master["selected_5000"] = (
#     master["id"].isin(selected_ids)
# )

# master["patient_projected"] = (
#     master["id"].isin(projected_map)
# )

# master["patient_restructured"] = (
#     master["id"].isin(patient_map)
# )

# master["patient_structured"] = (
#     master["patient"]
#     .apply(
#         lambda x: isinstance(x, dict) and bool(x)
#     )
# )

# master["priority_reevaluated"] = (
#     master["new_priority_full_input"]
#     .isin({"P1", "P2", "P3"})
# )


# # Contrôles stricts.

# assert len(master) == 18_523
# assert master["id"].is_unique

# assert master["selected_5000"].sum() == 5_000
# assert master["patient_projected"].sum() == 1_500
# assert master["patient_restructured"].sum() == 1_500

# print()
# print(f"✓ MASTER               : {len(master):,}")
# print(f"✓ Sélectionnés          : {master['selected_5000'].sum():,}")
# print(f"✓ EN projetés           : {master['patient_projected'].sum():,}")
# print(f"✓ EN restructurés       : {master['patient_restructured'].sum():,}")
# print(f"✓ Patient non vide      : {master['patient_structured'].sum():,}")
# print(f"✓ Priorité réévaluée    : {master['priority_reevaluated'].sum():,}")

In [ ]:
# audit_master = (
#     master
#     .groupby("language")
#     .agg(
#         records=("id", "size"),
#         patient_present=("patient_structured", "sum"),
#         projected=("patient_projected", "sum"),
#         restructured=("patient_restructured", "sum"),
#         reevaluated=("priority_reevaluated", "sum")
#     )
# )

# display(audit_master)

In [ ]:
# en_remaining = master.loc[
#     master["language"].eq("en")
#     & ~master["patient_projected"]
# ].copy()

# assert len(en_remaining) == 4_583

# print(f"EN restant à projeter : {len(en_remaining):,}")

In [171]:
# # Construit les 6 groupes langue × priorité réévaluée.

# assert len(master) == 18_523
# assert master["id"].is_unique

# PRIORITY_COL = "new_priority_full_input"

# assert PRIORITY_COL in master.columns
# assert master[PRIORITY_COL].isin(
#     {"P1", "P2", "P3"}
# ).all()


# # Crée la catégorie finale.

# master["new_category"] = (
#     master["language"].str.upper()
#     + "_"
#     + master[PRIORITY_COL]
# )


# # Les 6 morceaux.

# fr_p1 = master.loc[master["new_category"].eq("FR_P1")].copy()
# fr_p2 = master.loc[master["new_category"].eq("FR_P2")].copy()
# fr_p3 = master.loc[master["new_category"].eq("FR_P3")].copy()

# en_p1 = master.loc[master["new_category"].eq("EN_P1")].copy()
# en_p2 = master.loc[master["new_category"].eq("EN_P2")].copy()
# en_p3 = master.loc[master["new_category"].eq("EN_P3")].copy()


# # Contrôles.

# pieces = {
#     "FR_P1": fr_p1,
#     "FR_P2": fr_p2,
#     "FR_P3": fr_p3,
#     "EN_P1": en_p1,
#     "EN_P2": en_p2,
#     "EN_P3": en_p3
# }

# distribution = pd.Series(
#     {
#         name: len(df)
#         for name, df in pieces.items()
#     },
#     name="records"
# )

# display(distribution.to_frame())

# assert distribution.sum() == 18_523

# all_ids = [
#     record_id
#     for df in pieces.values()
#     for record_id in df["id"]
# ]

# assert len(all_ids) == 18_523
# assert len(set(all_ids)) == 18_523

# print(f"✓ Total : {distribution.sum():,}")
# print("✓ 6 groupes créés")
# print("✓ Aucun record perdu")
# print("✓ Aucun doublon entre les groupes")

In [172]:
# # Sauvegarde FR par priorité et EN par priorité + projection.

# SPLIT_DIR = OUTPUT_DIR / "priority_pools"
# SPLIT_DIR.mkdir(parents=True, exist_ok=True)


# # FR : 3 pools.

# fr_pools = {
#     "FR_P1": fr_p1,
#     "FR_P2": fr_p2,
#     "FR_P3": fr_p3
# }


# # EN : sépare projetés / non projetés.

# en_pools = {}

# for priority, df in {
#     "P1": en_p1,
#     "P2": en_p2,
#     "P3": en_p3
# }.items():

#     en_pools[f"EN_{priority}_projected"] = (
#         df.loc[df["patient_projected"]]
#         .copy()
#         .reset_index(drop=True)
#     )

#     en_pools[f"EN_{priority}_not_projected"] = (
#         df.loc[~df["patient_projected"]]
#         .copy()
#         .reset_index(drop=True)
#     )


# # Regroupe les 9 pools.

# pools = {
#     **fr_pools,
#     **en_pools
# }


# # Sauvegarde.

# for name, df in pools.items():
#     path = SPLIT_DIR / f"{name.lower()}.jsonl"

#     write_jsonl(
#         path,
#         df.to_dict("records")
#     )

#     print(
#         f"✓ {name:<22} "
#         f"{len(df):>6,} → {path}"
#     )


# # Contrôle intégrité.

# total = sum(
#     len(df)
#     for df in pools.values()
# )

# assert total == 18_523

# all_ids = [
#     record_id
#     for df in pools.values()
#     for record_id in df["id"]
# ]

# assert len(all_ids) == 18_523
# assert len(set(all_ids)) == 18_523

# print()
# print(f"✓ Total : {total:,}")
# print("✓ 9 pools")
# print("✓ Aucun record perdu")
# print("✓ Aucun doublon")

In [173]:
# # Recharge les pools utiles.

# SPLIT_DIR = OUTPUT_DIR / "priority_pools"


# def load_df(filename):
#     """Charge un JSONL dans un DataFrame."""
#     path = SPLIT_DIR / filename
#     df = pd.DataFrame(read_jsonl(path))

#     assert df["id"].is_unique

#     print(f"✓ {filename:<75} {len(df):>6,}")
#     return df


# # FR.

# fr_p1 = load_df("fr_p1.jsonl")
# fr_p2 = load_df("fr_p2.jsonl")
# fr_p3 = load_df("fr_p3.jsonl")


# # 1 500 EN projetés d'origine.

# en_p1_original = load_df(
#     "en_p1_projected.jsonl"
# )

# en_p2_original = load_df(
#     "en_p2_projected.jsonl"
# )

# # ⚠️ Ici il nous faut le fichier correspondant aux EN_P3
# # des 1 500 projetés d'origine, PAS celui des 4 479 recovered.
# en_p3_original = load_df(
#     "en_p3_projected.jsonl"
# )


# # 4 583 anciennement non projetés, maintenant entièrement traités.

# en_p1_recovered = load_df(
#     "en_p1_not_projected_projected_structured_reevaluated_v2.jsonl"
# )

# en_p2_recovered = load_df(
#     "en_p2_not_projected_projected_structured_reevaluated_v2.jsonl"
# )

# en_p3_recovered = load_df(
#     "en_p3_projected_structured_reevaluated_v2.jsonl"
# )

In [174]:
# # Fusionne les 6 pools EN.

# en_all = pd.concat(
#     [
#         en_p1_original,
#         en_p2_original,
#         en_p3_original,
#         en_p1_recovered,
#         en_p2_recovered,
#         en_p3_recovered
#     ],
#     ignore_index=True
# )

# # Contrôles d'intégrité.

# assert len(en_all) == 6_083
# assert en_all["id"].is_unique
# assert en_all["language"].eq("en").all()

# print(f"✓ EN total       : {len(en_all):,}")
# print(f"✓ IDs uniques    : {en_all['id'].nunique():,}")
# print("✓ Aucun doublon")

In [175]:
# print(en_all.columns.tolist())

# display(
#     en_all[
#         [
#             col
#             for col in en_all.columns
#             if "priority" in col.lower()
#         ]
#     ]
#     .notna()
#     .sum()
#     .rename("records")
#     .to_frame()
# )

In [176]:
# # Priorité finale EN :
# # nouvelle réévaluation projetée si disponible,
# # sinon réévaluation précédente des 1 500 EN.

# en_all["final_priority"] = (
#     en_all["new_priority_projected"]
#     .combine_first(
#         en_all["new_priority_full_input"]
#     )
#     .astype(str)
#     .str.upper()
#     .str.strip()
# )

# assert en_all["final_priority"].isin(
#     {"P1", "P2", "P3"}
# ).all()

# assert en_all["final_priority"].notna().all()

# print("✓ Priorité finale définie pour les 6 083 EN")

# display(
#     en_all["final_priority"]
#     .value_counts()
#     .reindex(["P1", "P2", "P3"])
#     .rename("records")
#     .to_frame()
# )

In [177]:
# # Découpe selon la priorité finale.

# en_p1_final = (
#     en_all.loc[
#         en_all["final_priority"].eq("P1")
#     ]
#     .copy()
#     .reset_index(drop=True)
# )

# en_p2_final = (
#     en_all.loc[
#         en_all["final_priority"].eq("P2")
#     ]
#     .copy()
#     .reset_index(drop=True)
# )

# en_p3_final = (
#     en_all.loc[
#         en_all["final_priority"].eq("P3")
#     ]
#     .copy()
#     .reset_index(drop=True)
# )


# # Contrôles.

# assert (
#     len(en_p1_final)
#     + len(en_p2_final)
#     + len(en_p3_final)
#     == 6_083
# )

# all_ids = pd.concat(
#     [
#         en_p1_final["id"],
#         en_p2_final["id"],
#         en_p3_final["id"]
#     ],
#     ignore_index=True
# )

# assert all_ids.is_unique

# print(f"✓ EN_P1 : {len(en_p1_final):,}")
# print(f"✓ EN_P2 : {len(en_p2_final):,}")
# print(f"✓ EN_P3 : {len(en_p3_final):,}")
# print(f"✓ Total : {len(all_ids):,}")

In [178]:
# # Sauvegarde les 3 pools EN définitifs.

# FINAL_DIR = OUTPUT_DIR / "priority_pools_final"
# FINAL_DIR.mkdir(parents=True, exist_ok=True)

# final_en_pools = {
#     "en_p1_final": en_p1_final,
#     "en_p2_final": en_p2_final,
#     "en_p3_final": en_p3_final
# }

# for name, df in final_en_pools.items():
#     path = FINAL_DIR / f"{name}.jsonl"

#     write_jsonl(
#         path,
#         df.to_dict("records")
#     )

#     print(f"✓ {name:<15} {len(df):>6,} → {path}")

In [179]:
# # Finalise les 3 pools FR.

# fr_p1_final = fr_p1.copy()
# fr_p2_final = fr_p2.copy()
# fr_p3_final = fr_p3.copy()

# fr_p1_final["final_priority"] = "P1"
# fr_p2_final["final_priority"] = "P2"
# fr_p3_final["final_priority"] = "P3"


# # Contrôles.

# assert len(fr_p1_final) == 1_670
# assert len(fr_p2_final) == 1_473
# assert len(fr_p3_final) == 9_297

# fr_all_ids = pd.concat(
#     [
#         fr_p1_final["id"],
#         fr_p2_final["id"],
#         fr_p3_final["id"]
#     ],
#     ignore_index=True
# )

# assert len(fr_all_ids) == 12_440
# assert fr_all_ids.is_unique

# print(f"✓ FR_P1 : {len(fr_p1_final):,}")
# print(f"✓ FR_P2 : {len(fr_p2_final):,}")
# print(f"✓ FR_P3 : {len(fr_p3_final):,}")
# print(f"✓ Total : {len(fr_all_ids):,}")

In [180]:
# # Sauvegarde le dataset consolidé final.

# SFT_CONSOLIDATED_PATH = (
#     OUTPUT_DIR / "sft_selected_5000_consolidated.jsonl"
# )

# write_jsonl(
#     SFT_CONSOLIDATED_PATH,
#     sft_consolidated.to_dict("records")
# )


# # Recharge le fichier réellement écrit.

# saved_consolidated = read_jsonl(
#     SFT_CONSOLIDATED_PATH
# )

# assert len(saved_consolidated) == 5_000
# assert len({r["id"] for r in saved_consolidated}) == 5_000

# print(f"✓ Sauvegardé : {SFT_CONSOLIDATED_PATH}")
# print(f"✓ Records    : {len(saved_consolidated):,}")
# print("✓ IDs uniques : 5 000")

In [181]:
# # Charge Qwen3-4B 4-bit pour l'inférence locale.

# MODEL_ID = "unsloth/Qwen3-4B-unsloth-bnb-4bit"

# model, tokenizer = FastModel.from_pretrained(
#     model_name=MODEL_ID,
#     max_seq_length=MAX_SEQ_LENGTH,
#     load_in_4bit=True,
#     full_finetuning=False
# )

# if tokenizer.pad_token_id is None:
#     tokenizer.pad_token = tokenizer.eos_token

# model.eval()

# print(f"✓ Modèle : {MODEL_ID}")
# print(f"✓ GPU    : {torch.cuda.get_device_name(0)}")

In [182]:
# TRIAGE_PROMPTS = {
#     "fr": """
# Tu es à l'entrée d'un service médical et tu dois évaluer la priorité
# de prise en charge du cas présenté.

# À partir UNIQUEMENT des informations fournies, attribue une priorité :

# P1 = très urgent : prise en charge immédiate ou très rapide.
# P2 = urgence modérée : prise en charge médicale rapide sans urgence vitale immédiate.
# P3 = urgence différée : situation stable ou peu urgente pouvant attendre.

# RÈGLES :
# - Évalue le cas présenté, pas uniquement la gravité générale de la maladie.
# - Utilise uniquement les informations explicitement disponibles.
# - N'invente aucune information.
# - N'utilise aucune priorité antérieure.
# - Ne cherche pas à reproduire une classification précédente.
# - Recherche en priorité les signes de gravité réellement présents.
# - Une maladie potentiellement grave ne justifie pas seule P1 si aucun
#   élément du cas n'indique une urgence immédiate.
# - L'absence d'information ne signifie pas que le signe est normal.
# - Si les informations sont limitées, attribue la priorité la mieux soutenue
#   par les éléments réellement disponibles.
# - L'explication doit être courte, factuelle et en français.
# - L'explication doit justifier directement la priorité choisie.

# Retourne uniquement un JSON valide, sans texte supplémentaire :
# {"priority": "P1", "explanation": "..."}
# """.strip(),

#     "en": """
# You are at the entrance of a medical department and must assess the
# priority of care for the presented case.

# Using ONLY the information provided, assign a priority:

# P1 = very urgent: immediate or very rapid medical care is required.
# P2 = moderate urgency: prompt medical care is required without evidence
# of an immediate life-threatening emergency.
# P3 = delayed urgency: stable or low-urgency situation where care can wait.

# RULES:
# - Assess the presented case, not only the general severity of the disease.
# - Use only explicitly available information.
# - Do not invent any information.
# - Do not use any previous priority.
# - Do not try to reproduce a previous classification.
# - Give particular attention to signs of severity actually present.
# - A potentially serious disease alone does not justify P1 if the case
#   contains no evidence of an immediate emergency.
# - Missing information must not be interpreted as a normal finding.
# - If information is limited, assign the priority best supported by the
#   information actually available.
# - The explanation must be short, factual and written in English.
# - The explanation must directly justify the selected priority.

# Return only valid JSON, without additional text:
# {"priority": "P1", "explanation": "..."}
# """.strip()
# }

# def build_triage_source(record: dict) -> dict:
#     """Construit le cas clinique sans ancienne décision."""
#     return {
#         "language": record.get("language"),
#         "question": record.get("question"),
#         "answer": record.get("answer"),
#         "model_context": record.get("model_context"),
#         "patient": record.get("patient") or {},
#         "medical_knowledge": record.get("medical_knowledge") or {}
#     }

In [183]:
# import gc
# import json

# gc.collect()

# def compact_source(value):
#     """Supprime les informations vides ou non renseignées."""
#     if isinstance(value, dict):
#         result = {}

#         for key, item in value.items():
#             cleaned = compact_source(item)

#             if cleaned not in (None, "", [], {}, "UNKNOWN"):
#                 result[key] = cleaned

#         return result

#     if isinstance(value, list):
#         return [
#             cleaned
#             for item in value
#             if (cleaned := compact_source(item))
#             not in (None, "", [], {}, "UNKNOWN")
#         ]

#     return value


# PATIENT_LLM_FIELDS = (
#     "age_years",
#     "sex",
#     "symptoms",
#     "condition_mentions",
#     "medical_history",
#     "duration",
#     "evolution",
#     "critical_signs",
#     "vital_signs"
# )

# KNOWLEDGE_LLM_FIELDS = (
#     "conditions",
#     "symptoms",
#     "complications",
#     "diagnostic_tests",
#     "laboratory_findings",
#     "imaging_findings",
#     "risk_factors",
#     "treatments",
#     "medications",
#     "procedures",
#     "pathogens",
#     "causes",
#     "anatomy"
# )


# def select_fields(data, fields):
#     """Conserve uniquement les informations cliniques utiles."""
#     if not isinstance(data, dict):
#         return {}

#     return {
#         field: data[field]
#         for field in fields
#         if field in data
#     }


# def build_triage_source(record: dict) -> dict:
#     """Construit une source clinique compacte sans ancienne priorité."""
#     source = {
#         "question": record.get("question"),
#         "patient": select_fields(
#             record.get("patient"),
#             PATIENT_LLM_FIELDS
#         ),
#         "medical_knowledge": select_fields(
#             record.get("medical_knowledge"),
#             KNOWLEDGE_LLM_FIELDS
#         )
#     }

#     return compact_source(source)

# def parse_json_response(response: str) -> dict:
#     """Extrait le premier objet JSON valide."""
#     response = response.strip()
#     decoder = json.JSONDecoder()

#     for i, char in enumerate(response):
#         if char != "{":
#             continue

#         try:
#             result, _ = decoder.raw_decode(response[i:])
#             return result
#         except json.JSONDecodeError:
#             continue

#     raise ValueError(
#         f"JSON introuvable :\n{response[:1000]}"
#     )


# def generate_triage(record: dict) -> dict:
#     """Génère priorité et explication dans la langue du record."""
#     language = str(record.get("language", "")).lower()

#     if language not in TRIAGE_PROMPTS:
#         raise ValueError(
#             f"Langue invalide : {record['id']} → {language}"
#         )

#     source = build_triage_source(record)

#     prompt = (
#         f"{TRIAGE_PROMPTS[language]}\n\n"
#         f"{'CAS À ÉVALUER' if language == 'fr' else 'CASE TO ASSESS'}:\n"
#         f"{json.dumps(source, ensure_ascii=False)}"
#     )

#     text = tokenizer.apply_chat_template(
#         [{"role": "user", "content": prompt}],
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=False
#     )

#     inputs = tokenizer(
#         text,
#         return_tensors="pt",
#         truncation=False
#     )

#     input_length = inputs["input_ids"].shape[1]
#     max_input_length = MAX_SEQ_LENGTH - MAX_NEW_TOKENS

#     if input_length > max_input_length:
#         raise ValueError(
#             f"Prompt trop long : {record['id']} → "
#             f"{input_length}/{max_input_length} tokens"
#         )

#     inputs = inputs.to(model.device)

#     with torch.inference_mode():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=MAX_NEW_TOKENS,
#             do_sample=False,
#             use_cache=True,
#             eos_token_id=tokenizer.eos_token_id,
#             pad_token_id=tokenizer.pad_token_id
#         )

#     generated = outputs[
#         0,
#         inputs["input_ids"].shape[1]:
#     ]

#     response = tokenizer.decode(
#         generated,
#         skip_special_tokens=True
#     ).strip()

#     result = parse_json_response(response)

#     return {
#         "priority": result.get("priority"),
#         "explanation": result.get("explanation")
#     }

In [184]:
# # Test équilibré : 5 records par catégorie.

# test_30 = (
#     sft_selected
#     .groupby("category", group_keys=False)
#     .head(5)
#     .reset_index(drop=True)
# )

# assert len(test_30) == 30

# display(
#     test_30["category"]
#     .value_counts()
#     .sort_index()
# )

# sft_selected = test_30

In [185]:
# TEMP_PATH = OUTPUT_DIR / "sft_triage_temp.jsonl"

# print(f"Fichier provisoire : {TEMP_PATH}")

# enriched_records = []

# with TEMP_PATH.open("w", encoding="utf-8") as f:
#     for record in tqdm(
#         sft_selected.to_dict("records"),
#         desc="Nouveau triage SFT",
#         unit="record"
#     ):
#         result = generate_triage(record)

#         priority = str(
#             result.get("priority", "")
#         ).upper().strip()

#         explanation = str(
#             result.get("explanation", "")
#         ).strip()

#         # Contrôle la sortie LLM.
#         if priority not in {"P1", "P2", "P3"}:
#             raise RuntimeError(
#                 f"Priorité invalide : {record['id']} → {priority}"
#             )

#         if not explanation:
#             raise RuntimeError(
#                 f"Explication absente : {record['id']}"
#             )

#         enriched = {
#             **record,
#             "new_priority": priority,
#             "new_explanation": explanation
#         }

#         enriched_records.append(enriched)

#         # Sauvegarde immédiatement le résultat.
#         f.write(
#             json.dumps(
#                 enriched,
#                 ensure_ascii=False
#             ) + "\n"
#         )
#         f.flush()

In [186]:
# test_results = pd.DataFrame(enriched_records)

# display(
#     test_results[
#         [
#             "id",
#             "language",
#             "category",
#             "priority",
#             "new_priority",
#             "explanation",
#             "new_explanation"
#         ]
#     ]
# )

In [187]:
# OUTPUT_PATH = (
#     OUTPUT_DIR
#     / "sft_selected_5000_llm_review.jsonl"
# )

# write_jsonl(
#     OUTPUT_PATH,
#     enriched_records
# )

# print(f"✓ Export : {OUTPUT_PATH.resolve()}")

In [195]:
# Charge le SFT final et crée les splits stratifiés 80/10/10.

SFT_PATH = OUTPUT_DIR / "sft_final_5000.jsonl"

SFT_COLUMNS = [
    "id", "language", "source_split",
    "model_context", "explanation", "priority", "context_hash"
]

split_paths = {
    name: RUN_SPLIT_DIR / f"sft_{name}.jsonl"
    for name in ("train", "validation", "test")
}


# Charge et contrôle le dataset final.

sft_df = pd.DataFrame(read_jsonl(SFT_PATH))

assert len(sft_df) == 5_000
assert sft_df["id"].notna().all() and sft_df["id"].is_unique
assert sft_df["language"].isin({"fr", "en"}).all()
assert sft_df["priority"].isin({1, 2, 3}).all()
assert sft_df["model_context"].fillna("").str.strip().ne("").all()
assert sft_df["explanation"].fillna("").str.strip().ne("").all()
assert sft_df["context_hash"].notna().all()
assert sft_df["context_hash"].is_unique

print(f"✓ SFT final chargé : {len(sft_df):,}")

✓ SFT final chargé : 5,000


In [198]:
# Crée les splits stratifiés langue × priorité.

strata = (
    sft_df["language"].str.upper()
    + "_P"
    + sft_df["priority"].astype(str)
)

train_df, temp_df = train_test_split(
    sft_df,
    test_size=1_000,
    stratify=strata,
    random_state=RANDOM_SEED
)

temp_strata = (
    temp_df["language"].str.upper()
    + "_P"
    + temp_df["priority"].astype(str)
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=500,
    stratify=temp_strata,
    random_state=RANDOM_SEED
)

splits = {
    "train": train_df.reset_index(drop=True),
    "validation": validation_df.reset_index(drop=True),
    "test": test_df.reset_index(drop=True)
}

for name, df in splits.items():
    write_jsonl(
        split_paths[name],
        df[SFT_COLUMNS].to_dict("records")
    )


# Vérifie volumes, population et absence de fuite.

split_ids = {
    name: set(df["id"].astype(str))
    for name, df in splits.items()
}

split_hashes = {
    name: set(df["context_hash"])
    for name, df in splits.items()
}

assert [len(splits[name]) for name in ("train", "validation", "test")] == [
    4_000, 500, 500
]

assert not split_ids["train"] & split_ids["validation"]
assert not split_ids["train"] & split_ids["test"]
assert not split_ids["validation"] & split_ids["test"]

assert set().union(*split_ids.values()) == set(sft_df["id"].astype(str))

for left, right in (
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
):
    assert not split_hashes[left] & split_hashes[right]


# Affiche la distribution finale.

summary = pd.concat({
    name: pd.crosstab(df["priority"], df["language"])
    for name, df in splits.items()
})

display(summary)

for name, df in splits.items():
    counts = df["priority"].value_counts().reindex([1, 2, 3], fill_value=0)

    print(
        f"{name:10} | {len(df):4} | "
        f"P1={counts[1]:4} P2={counts[2]:4} P3={counts[3]:4} | "
        f"FR={(df['language'] == 'fr').sum():4} "
        f"EN={(df['language'] == 'en').sum():4}"
    )

print("\n✓ Splits SFT créés et figés")
print("✓ Aucune fuite entre les splits")
print(f"✓ Source : {SFT_PATH}")
print(f"✓ Sortie : {RUN_SPLIT_DIR}")

language              en   fr
           priority          
train      1         667  667
           2         666  666
           3         667  667
validation 1          83   83
           2          84   84
           3          83   83
test       1          84   84
           2          83   83
           3          83   83

train      | 4000 | P1=1334 P2=1332 P3=1334 | FR=2000 EN=2000
validation |  500 | P1= 166 P2= 168 P3= 166 | FR= 250 EN= 250
test       |  500 | P1= 168 P2= 166 P3= 166 | FR= 250 EN= 250

✓ Splits SFT créés et figés
✓ Aucune fuite entre les splits
✓ Source : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\sft_dataset_creation\sft_final_5000.jsonl
✓ Sortie : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\sft_dataset_creation\splits
